# Lab 17: Medidas POVM y Proyectores

La medida cuántica más general se describe mediante un **Operador de Medida con Valor Positivo** (POVM): un conjunto $\{M_k\}$ con $M_k \geq 0$ y $\sum_k M_k = I$.

La probabilidad del resultado $k$ es $p_k = \text{Tr}(M_k \rho)$.

Casos especiales:
- **Proyectores von Neumann**: $M_k = P_k = |k\rangle\langle k|$, que son ortogonales y satisfacen $P_k^2 = P_k$.
- **POVM no proyectivo**: permite distinguir estados no ortogonales con mayor eficiencia informacional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit.primitives import StatevectorSampler

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)

# Estados relevantes
sv0 = Statevector.from_label('0')
sv1 = Statevector.from_label('1')
svp = Statevector.from_label('+')
svm = Statevector.from_label('-')

print("Estados de prueba: |0⟩, |1⟩, |+⟩, |−⟩")

## 1. Medida Proyectiva en Base Computacional

Los proyectores en la base $\{|0\rangle, |1\rangle\}$ son $P_0 = |0\rangle\langle 0|$ y $P_1 = |1\rangle\langle 1|$.

Verificamos que:
- Son ortogonales: $P_0 P_1 = 0$
- Son completos: $P_0 + P_1 = I$
- Son idempotentes: $P_k^2 = P_k$

In [ ]:
# Proyectores en base Z
P0 = np.outer([1,0],[1,0]).astype(complex)
P1 = np.outer([0,1],[0,1]).astype(complex)

print("Proyectores base Z:")
print(f"  P0·P1 = 0:   {np.allclose(P0 @ P1, 0)}")
print(f"  P0+P1 = I:   {np.allclose(P0 + P1, I)}")
print(f"  P0² = P0:    {np.allclose(P0 @ P0, P0)}")
print()

# Probabilidades para distintos estados
states = {'|0⟩': sv0.data, '|1⟩': sv1.data, '|+⟩': svp.data, '|−⟩': svm.data}
print(f"{'Estado':>6} | {'p(0)':>8} | {'p(1)':>8}")
print("-" * 30)
for name, psi in states.items():
    rho = np.outer(psi, psi.conj())
    p0 = np.trace(P0 @ rho).real
    p1 = np.trace(P1 @ rho).real
    print(f"{name:>6} | {p0:>8.4f} | {p1:>8.4f}")

# Proyectores en base X
Pp = np.outer([1,1],[1,1])/2  # |+⟩⟨+|
Pm = np.outer([1,-1],[1,-1])/2  # |−⟩⟨−|
print()
print("Medida en base X:")
for name, psi in states.items():
    rho = np.outer(psi, psi.conj())
    pp = np.trace(Pp @ rho).real
    pm = np.trace(Pm @ rho).real
    print(f"{name:>6} | p(+)={pp:.4f} | p(−)={pm:.4f}")

## 2. POVM para Distinguir Estados No Ortogonales

El **trine POVM** permite distinguir con probabilidad óptima entre los estados $|\psi_0\rangle = |0\rangle$, $|\psi_1\rangle = (-|0\rangle + \sqrt{3}|1\rangle)/2$, $|\psi_2\rangle = (-|0\rangle - \sqrt{3}|1\rangle)/2$ (vértices del tríangulo de Bloch).

Los elementos POVM $\Pi_k = \frac{2}{3}|\psi_k^\perp\rangle\langle\psi_k^\perp|$ son no proyectivos y satisfacen $\sum_k \Pi_k = I$.

In [ ]:
# Trine POVM: los 3 estados del triángulo ecuatorial de la esfera de Bloch
angles = [0, 2*np.pi/3, 4*np.pi/3]
trine_states = [np.array([np.cos(a/2), np.sin(a/2)], dtype=complex) for a in angles]

# Elementos POVM (proyectores escalados)
Pi = [(2/3) * np.outer(psi, psi.conj()) for psi in trine_states]

# Verificar completitud
total = sum(Pi)
print("Completitud POVM ∑Πk = I:", np.allclose(total, I))
print(f"Error: {np.linalg.norm(total - I):.2e}")

# Probabilidades de distinguir correctamente
print("
Matriz de probabilidades p(resultado k | estado enviado j):")
print(f"{'':>12}", end="")
for k in range(3): print(f" |ψ_{k}⟩", end="")
print()
for k in range(3):
    print(f"Resultado Π_{k} | ", end="")
    for j in range(3):
        rho_j = np.outer(trine_states[j], trine_states[j].conj())
        p = np.trace(Pi[k] @ rho_j).real
        print(f" {p:.3f}", end="")
    print()

# Comparación con medida proyectiva en base Z
print("
Medida Z sobre estados trine:")
for j, psi in enumerate(trine_states):
    rho = np.outer(psi, psi.conj())
    print(f"  |ψ_{j}⟩: p(0)={np.trace(P0@rho).real:.3f}, p(1)={np.trace(P1@rho).real:.3f}")

## 3. Medida Tomográfica — Reconstrucción de Estado

La **tomografía cuántica de estados** reconstruye $\rho$ a partir de valores esperados de observables. Para 1 qubit basta medir $\langle X\rangle$, $\langle Y\rangle$, $\langle Z\rangle$:

$$\rho = \frac{1}{2}(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z)$$

In [ ]:
def tomography_1qubit(state_vec, shots=2048):
    """Reconstruye ρ de 1 qubit mediante medidas en X, Y, Z."""
    sampler = StatevectorSampler()
    expectations = {}
    for axis, basis_change in [('Z', None), ('X', 'h'), ('Y', 'sdg+h')]:
        qc = QuantumCircuit(1, 1)
        if isinstance(state_vec, np.ndarray):
            qc.initialize(state_vec, 0)
        if axis == 'X':
            qc.h(0)
        elif axis == 'Y':
            qc.sdg(0); qc.h(0)
        qc.measure(0, 0)
        job = sampler.run([qc], shots=shots)
        counts = job.result()[0].data.c.get_counts()
        p0 = counts.get('0', 0) / shots
        p1 = counts.get('1', 0) / shots
        expectations[axis] = p0 - p1
    rho_rec = 0.5*(I + expectations['X']*X + expectations['Y']*Y + expectations['Z']*Z)
    return rho_rec, expectations

# Reconstruir varios estados
test_states = {
    '|0⟩': sv0.data, '|+⟩': svp.data,
    '|i⟩': np.array([1, 1j])/np.sqrt(2)
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, psi) in zip(axes, test_states.items()):
    rho_ideal = np.outer(psi, psi.conj())
    rho_rec, exp = tomography_1qubit(psi)
    fid = float(state_fidelity(DensityMatrix(rho_ideal), DensityMatrix(rho_rec)))
    im = ax.imshow(np.abs(rho_rec), cmap='Blues', vmin=0, vmax=0.6)
    ax.set_title(f'{name}  F={fid:.4f}', fontsize=11)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['|0⟩','|1⟩']); ax.set_yticklabels(['⟨0|','⟨1|'])
    ax.set_xlabel(f'⟨X⟩={exp["X"]:.3f}, ⟨Y⟩={exp["Y"]:.3f}, ⟨Z⟩={exp["Z"]:.3f}', fontsize=8)

plt.suptitle('Tomografía de estado: |ρ_reconstruida|', fontsize=12)
plt.tight_layout(); plt.show()

## 4. Teorema de Naimark: POVM como Proyección en Espacio Ampliado

Todo POVM puede implementarse como una **medida proyectiva** en un espacio de Hilbert mayor (espacio del sistema + ancilla). Este teorema conecta la teoría de medidas generalizadas con la medida estándar von Neumann.

In [ ]:
# Implementar el POVM del trine como medida proyectiva con ancilla
# Para el caso de 2 elementos (POVM de discriminación de estados)
# POVM: Π+ = (1+1/√2)|+⟩⟨+|, Π- = (1-1/√2)|−⟩⟨−|, Π? = I - Π+ - Π-
# Usamos el POVM de discriminación óptima para |0⟩ vs |+⟩
eta = np.pi/8  # ángulo de Helstrom para distinguir |0⟩ y |+⟩

# Probabilidad de error para discriminación óptima
psi_0 = sv0.data
psi_1 = svp.data
overlap = abs(np.vdot(psi_0, psi_1))
p_error_helstrom = (1 - np.sqrt(1 - overlap**2)) / 2
p_error_z = 0.5 * (1 - abs(psi_0[0])**2 + abs(psi_1[0])**2) / 2

print("Discriminación |0⟩ vs |+⟩:")
print(f"  Solapamiento: |⟨0|+⟩| = {overlap:.4f}")
print(f"  P_error (Helstrom óptimo): {p_error_helstrom:.4f}")
print(f"  P_error (medida Z simple): {0.5*(1 - (1-1/np.sqrt(2))):.4f}")
print()
print("Principio de Holevo: la POVM óptima puede mejorar la discriminación")
print("de estados no ortogonales sobre cualquier medida proyectiva.")

# Verificación del bound de Helstrom
for rho_0, rho_1, label in [
    (np.outer(sv0.data, sv0.data.conj()),
     np.outer(svp.data, svp.data.conj()),
     '|0⟩ vs |+⟩'),
    (np.outer(sv0.data, sv0.data.conj()),
     np.outer(sv1.data, sv1.data.conj()),
     '|0⟩ vs |1⟩ (ortogonales)'),
]:
    evals = np.linalg.eigvalsh(0.5*rho_0 - 0.5*rho_1)
    p_err = 0.5 * (1 - sum(abs(e) for e in evals))
    print(f"  {label}: P_error mínima = {p_err:.4f}")